In [ ]:
%matplotlib inline
import torch
from conformal.specbridge import mass_spec_gym_candidates, mass_spec_gym_dataset

# 1. Precompute Model Predictions

**This takes a long time ! Run it once, and then reuse the saved CSV predictions**

Load datasets

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = mass_spec_gym_dataset()
candidates = mass_spec_gym_candidates()

Load model

In [ ]:
from argparse import Namespace
from specbridge.adapters.dreams_adapter import load_dreams_encoder
from specbridge.models.mapper import DreamsToMolCondition
from conformal.specbridge import SmilesPredictor


dreams_encoder = load_dreams_encoder(
    "Models/SpecBridge/runs/DreaMS/ssl_model.ckpt", d_in=2048, d_out=1024
)

dtmc = DreamsToMolCondition(
    dreams_encoder,
    d_out=2048,
    mapper_hidden=2048,
    gaussian=False,
    mol_space="chemberta",
    chemberta_model="Derify/ChemBERTa_augmented_pubchem_13m",
    args=Namespace(n_blocks=8),
).eval()

dtmc.load_state_dict(
    torch.load(
        "Models/SpecBridge/runs/msgym/SpecBridge_MSGYM_checkpoint.pt",
        map_location="cpu",
        weights_only=False,
    )["model"],
    strict=False,
)

for p in dtmc.parameters():
    p.requires_grad = False


model = SmilesPredictor(dtmc, device)

Smiles prediction example

In [ ]:
spectrum = dataset[10]
ground_truth = spectrum["smiles"]

print(f"Sample: {spectrum['title']}")
print(f"True SMILES: {ground_truth}")
print(f"Number of candidates: {len(candidates[ground_truth])}")

predictions = model.forward(spectrum, candidates[ground_truth], top_k=5)


print("\nTop 5 Predictions:")
for i, (smiles, score) in enumerate(predictions, 1):
    is_correct = "✓" if smiles == ground_truth else "✗"
    print(f"{i}. {is_correct} {smiles[:60]} | score: {score:.4f}")

Run top-1 predictions on the test set

In [ ]:
import csv
from tqdm import tqdm

with open("predictions.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Ground Truth", "Prediction"])

    for sample in tqdm(iter(dataset), total=len(dataset)):
        sample_candidates = candidates[sample["smiles"]]

        (prediction, _), *_ = model.forward(sample, sample_candidates)
        writer.writerow([sample["smiles"], prediction])

# 2. Calibration

### Standard Conformal Predictions

In [ ]:
%matplotlib inline
import numpy as np
import polars as pl

from conformal.regression import FixedRegressor
from conformal.fgw import FGW
from conformal.graph import Graph
from conformal.metrics import Metrics
from conformal.specbridge import mass_spec_gym_candidates

fgw = FGW(cost="laplacian")
regressor = FixedRegressor(target=0.9)

df = pl.read_csv("predictions.csv")

candidates = mass_spec_gym_candidates()
truths = df["Ground Truth"].to_numpy()
preds = df["Prediction"].to_numpy()

Fit the calibration model (takes a few seconds)

In [ ]:
from tqdm import tqdm

calibration_rate = 0.6
rng = np.random.default_rng(seed=42)
mask = rng.random(len(df)) < calibration_rate

calib_truths = truths[mask]
calib_preds = preds[mask]

distances: list[float] = []

# Compute prediction non-conformity scores
for pred, truth in tqdm(zip(calib_preds, calib_truths), total=len(calib_truths)):
    try:
        g1 = Graph.from_smiles(pred)
        g2 = Graph.from_smiles(truth)
        distances.append(fgw(g1, g2))
    except Exception:
        pass  # disconnected molecule, ignore


# Fit the predictor
regressor.fit(distances)

print(f"Predictor threshold: {regressor._threshold:.3f}")

Evaluate the model (takes 30 min to run)

In [ ]:
valid_truths = truths[~mask]
valid_preds = preds[~mask]

# Accumulate metrics
correct_coverage: list[bool] = []
candidate_sizes: list[int] = []
conformal_sizes: list[int] = []

for pred, truth in tqdm(zip(valid_preds, valid_truths), total=len(valid_preds)):
    try:
        g_pred = Graph.from_smiles(pred)
        g_truth = Graph.from_smiles(truth)
    except Exception:
        continue  # disconnected molecule, ignore

    cands = candidates[truth]
    threshold = regressor.threshold()
    candidate_size = 0
    conformal_size = 0

    # See if the ground truth is in the conformal set
    correct_coverage.append(fgw(g_pred, g_truth) <= threshold)

    for cand in cands:
        try:
            g_cand = Graph.from_smiles(cand)
        except Exception:
            continue  # disconnected molecule, ignore

        candidate_size += 1
        if fgw(g_pred, g_cand) <= threshold:
            conformal_size += 1

    candidate_sizes.append(candidate_size)
    conformal_sizes.append(conformal_size)

metrics = Metrics(correct_coverage, candidate_sizes, conformal_sizes)


# save to disk for later analysis
metrics.save("default-metrics.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")


In [ ]:
metrics.plot_binned_coverage_rate()

In [ ]:
metrics.plot_set_sizes()

# 3. Score Conformal Quantile Regression

### Calibration from candidate sizes

Exact same code, except a few lines to get the candidate size.

In [ ]:
%matplotlib inline
import numpy as np
import polars as pl

from conformal.regression import CandidateSizeRegressor
from conformal.fgw import FGW
from conformal.graph import Graph
from conformal.metrics import Metrics
from conformal.specbridge import mass_spec_gym_candidates

fgw = FGW(cost="laplacian")
regressor = CandidateSizeRegressor(target=0.9, kind="linear")

df = pl.read_csv("predictions.csv")

candidates = mass_spec_gym_candidates()
truths = df["Ground Truth"].to_numpy()
preds = df["Prediction"].to_numpy()

Fit the calibration model (takes a few seconds)

In [ ]:
from tqdm import tqdm

calibration_rate = 0.6
rng = np.random.default_rng(seed=42)
mask = rng.random(len(df)) < calibration_rate

calib_truths = truths[mask]
calib_preds = preds[mask]

distances: list[float] = []
candidate_sizes: list[int] = []

# Compute prediction non-conformity scores
for pred, truth in tqdm(zip(calib_preds, calib_truths), total=len(calib_truths)):
    try:
        g1 = Graph.from_smiles(pred)
        g2 = Graph.from_smiles(truth)
        distances.append(fgw(g1, g2))
        candidate_sizes.append(len(candidates[truth]))
    except Exception:
        pass  # disconnected molecule, ignore


# Fit the predictor
regressor.fit(distances, candidate_sizes)

print(f"Predictor threshold: {regressor._threshold:.3f}")

Evaluate the model (takes 30 min to run)

In [ ]:
valid_truths = truths[~mask]
valid_preds = preds[~mask]

# Accumulate metrics
correct_coverage: list[bool] = []
candidate_sizes: list[int] = []
conformal_sizes: list[int] = []

for pred, truth in tqdm(zip(valid_preds, valid_truths), total=len(valid_preds)):
    try:
        g_pred = Graph.from_smiles(pred)
        g_truth = Graph.from_smiles(truth)
    except Exception:
        continue  # disconnected molecule, ignore

    cands = candidates[truth]
    threshold = regressor.threshold(len(cands))
    candidate_size = 0
    conformal_size = 0

    # See if the ground truth is in the conformal set
    correct_coverage.append(fgw(g_pred, g_truth) <= threshold)

    for cand in cands:
        try:
            g_cand = Graph.from_smiles(cand)
        except Exception:
            continue  # disconnected molecule, ignore

        candidate_size += 1
        if fgw(g_pred, g_cand) <= threshold:
            conformal_size += 1

    candidate_sizes.append(candidate_size)
    conformal_sizes.append(conformal_size)

metrics = Metrics(correct_coverage, candidate_sizes, conformal_sizes)


# save to disk for later analysis
metrics.save("scqr-metrics.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")


### SCQR from mass spectrum embeddings

First, precompute the mass spectrum embeddings.

In [ ]:
from tqdm import tqdm

embeddings = torch.zeros((len(dataset), 768))

for i, sample in tqdm(enumerate(iter(dataset)), total=len(dataset)):
    embedding = model.spectrum_embedding(sample)
    embeddings[i] = embedding[0].cpu()

torch.save(embeddings, "spectra-embeddings.pkl")

In [ ]:
%matplotlib inline
import numpy as np
import polars as pl
import torch
from torch import Tensor

from conformal.regression import EmbeddingRegressor
from conformal.fgw import FGW
from conformal.graph import Graph
from conformal.metrics import Metrics
from conformal.specbridge import mass_spec_gym_candidates

embeddings: Tensor = torch.load("spectra-embeddings.pkl", weights_only=False)
fgw = FGW(cost="laplacian")
regressor = EmbeddingRegressor(target=0.9, embed_dim=embeddings.shape[1])

df = pl.read_csv("predictions.csv")

candidates = mass_spec_gym_candidates()
truths = df["Ground Truth"].to_numpy()
preds = df["Prediction"].to_numpy()

Fit the calibration model (takes a few seconds)

In [ ]:
from tqdm import tqdm

calibration_rate = 0.6
rng = np.random.default_rng(seed=42)
mask = rng.random(len(df)) < calibration_rate

calib_truths = truths[mask]
calib_preds = preds[mask]
calib_embeds = embeddings[mask]  # type: ignore

distances: list[float] = []
valid_molec_mask = np.ones(len(calib_truths), dtype=np.bool)

# Compute prediction non-conformity scores
for i, (pred, truth) in tqdm(
    enumerate(zip(calib_preds, calib_truths)), total=len(calib_truths)
):
    try:
        g1 = Graph.from_smiles(pred)
        g2 = Graph.from_smiles(truth)
        distances.append(fgw(g1, g2))
    except Exception:
        valid_molec_mask[i] = False

# Reindex the mass spectrum embeddings to remove those that correspond to
# bad molecules
calib_embeds = calib_embeds[valid_molec_mask]  # type: ignore

assert len(calib_embeds) == len(distances)

# Fit the predictor
regressor.fit(distances, calib_embeds)

print(f"Predictor threshold: {regressor._threshold:.3f}")

Evaluate the model (takes 30 min to run)

In [ ]:
valid_truths = truths[~mask]
valid_preds = preds[~mask]
valid_embeds = embeddings[~mask] # type: ignore

# Accumulate metrics
correct_coverage: list[bool] = []
candidate_sizes: list[int] = []
conformal_sizes: list[int] = []

for i, (pred, truth) in tqdm(enumerate(zip(valid_preds, valid_truths)), total=len(valid_preds)):
    try:
        g_pred = Graph.from_smiles(pred)
        g_truth = Graph.from_smiles(truth)
    except Exception:
        continue  # disconnected molecule, ignore

    cands = candidates[truth]
    threshold = regressor.threshold(valid_embeds[i])
    candidate_size = 0
    conformal_size = 0

    # See if the ground truth is in the conformal set
    correct_coverage.append(fgw(g_pred, g_truth) <= threshold)

    for cand in cands:
        try:
            g_cand = Graph.from_smiles(cand)
        except Exception:
            continue  # disconnected molecule, ignore

        candidate_size += 1
        if fgw(g_pred, g_cand) <= threshold:
            conformal_size += 1

    candidate_sizes.append(candidate_size)
    conformal_sizes.append(conformal_size)

metrics = Metrics(correct_coverage, candidate_sizes, conformal_sizes)


# save to disk for later analysis
metrics.save("default-metrics.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")
